# 03 — Intervention Coding

Port the Regan (2002) intervention variable onto each of the 25 directed-dyad files.

**Data source**: `data/raw/regan/replication.10.26.01.dta`

**Reference R script**: `zzz-old_version/Paper-Shadow/R/08-addInterventions.R`

**Output**: `data/interim/dd_int_{cy}_{ud}.parquet` for cy ∈ 1–5, ud ∈ 1–5 (25 files).

**Intervention coding**:
- `NaN` — non-onset rows (no civil war onset in host country)
- `0`   — onset row, no military intervention by this potential intervener
- `1`   — onset row, government-biased military intervention
- `2`   — onset row, opposition-biased military intervention

**Year-window matching** (verbatim from R `inRange(year, int$year[i], ub=5, lb=2)`):
An intervention in year V matches an onset in year Y if V − 2 ≤ Y ≤ V + 5.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from shadow.data.interventions import build_intervention_table, code_interventions

warnings.filterwarnings("ignore")

RAW     = Path("../data/raw")
INTERIM = Path("../data/interim")

N_IMP_CY = 5
N_IMP_UD = 5

print("Setup complete.")

Setup complete.


## § 1 — Build Intervention Lookup Table

Load and clean the Regan `.dta` file: apply per-conflict coding corrections,
standardise ccodes, convert 2-digit years, and deduplicate to one row per
unique (host, intervener, year) military intervention.

In [2]:
int_table = build_intervention_table(RAW / "regan" / "replication.10.26.01.dta")

print(f"Intervention table: {len(int_table):,} unique (host, intervener, year) military interventions")
print(f"  Government-biased (target=1): {(int_table.target == 1).sum()}")
print(f"  Opposition-biased (target=2): {(int_table.target == 2).sum()}")
print(f"  Unique intervening states:    {int_table.ccode_B.nunique()}")
print(f"  Unique host states:           {int_table.ccode_A.nunique()}")
print(f"  Year range:                   {int_table.year.min()}–{int_table.year.max()}")
print()
print("Sample rows:")
int_table.head(10)

Intervention table: 424 unique (host, intervener, year) military interventions
  Government-biased (target=1): 252
  Opposition-biased (target=2): 172
  Unique intervening states:    67
  Unique host states:           55
  Year range:                   1944–1999

Sample rows:


,ccode_A,ccode_B,year,target,ddyear
0,042,002,1965,1,042_002_1965
1,090,002,1954,2,090_002_1954
2,090,002,1966,1,090_002_1966
3,090,002,1971,1,090_002_1971
4,090,002,1977,1,090_002_1977
5,090,002,1981,1,090_002_1981
6,090,002,1983,1,090_002_1983
7,090,002,1984,1,090_002_1984
8,090,002,1990,1,090_002_1990
9,090,002,1994,1,090_002_1994


## § 2 — Main Loop: Code Interventions onto All 25 DD Files

In [3]:
for i_cy in range(1, N_IMP_CY + 1):
    for i_ud in range(1, N_IMP_UD + 1):
        in_path  = INTERIM / f"dd_{i_cy}_{i_ud}.parquet"
        out_path = INTERIM / f"dd_int_{i_cy}_{i_ud}.parquet"

        dd     = pd.read_parquet(in_path)
        dd_int = code_interventions(dd, int_table)
        dd_int.to_parquet(out_path, index=False)

        n_onset   = (dd_int["onset_A"] == 1).sum()
        n_gov     = (dd_int["intervention"] == 1).sum()
        n_opp     = (dd_int["intervention"] == 2).sum()
        print(
            f"  dd_int_{i_cy}_{i_ud}: {len(dd_int):,} rows  "
            f"| onset rows: {n_onset:,}  "
            f"| gov: {n_gov}  opp: {n_opp}"
        )

print(f"\nAll {N_IMP_CY * N_IMP_UD} dd_int files written.")

  dd_int_1_1: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_1_2: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_1_3: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_1_4: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_1_5: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


...

  dd_int_5_1: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_5_2: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_5_3: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_5_4: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91


  dd_int_5_5: 1,198,730 rows  | onset rows: 25,728  | gov: 128  opp: 91

All 25 dd_int files written.


## § 3 — Validation

Checks run against `dd_int_1_1` as a representative file.

**Expected from original Regan / Fearon-Laitin pipeline**  
(Note: our updated onset data covers 1946–2014 vs FL's 111 wars, so
counts will differ — direction and magnitude should be similar.)
- Total military interventions in int_table: 118 (original FL; we may have more)
- Government-biased: 66, Opposition-biased: 52
- Unique intervening states: 53
- Wars with no intervention: 59 / 111

In [4]:
import glob as _glob

# ── File inventory ─────────────────────────────────────────────────────────
dd_int_files = sorted(_glob.glob(str(INTERIM / "dd_int_*.parquet")))
print(f"DD_INT files produced: {len(dd_int_files)}  (expected {N_IMP_CY * N_IMP_UD})")
assert len(dd_int_files) == N_IMP_CY * N_IMP_UD, "Wrong number of output files!"

# ── Spot-check dd_int_1_1 ─────────────────────────────────────────────────
sample = pd.read_parquet(INTERIM / "dd_int_1_1.parquet")
onset  = sample[sample["onset_A"] == 1]

n_onset_wars   = onset.drop_duplicates(["ccode_A", "year"]).shape[0]
n_wars_int     = onset[onset["intervention"] != 0].ccode_A.nunique()
n_wars_no_int  = onset[onset["intervention"] == 0].ccode_A.nunique()
n_uniq_states  = onset[onset["intervention"] != 0].ccode_B.nunique()

print()
print("dd_int_1_1 — onset rows only:")
print(f"  Unique onset country-years:           {n_onset_wars}")
print(f"  Unique civil war hosts:               {onset.ccode_A.nunique()}")
print(f"  Wars with any intervention:           {n_wars_int}")
print(f"  Wars with no intervention:            {n_wars_no_int}")
print(f"  Unique intervening states:            {n_uniq_states}")
print(f"  Onset rows coded gov-biased (=1):     {(onset.intervention == 1).sum()}")
print(f"  Onset rows coded opp-biased (=2):     {(onset.intervention == 2).sum()}")
print(f"  Onset rows coded no-intervention (=0):{(onset.intervention == 0).sum()}")
print()

# ── Known spot-checks ─────────────────────────────────────────────────────
# USA (002) in Guatemala (090): gov-biased in multiple onset years
ug = sample[(sample.ccode_A == "090") & (sample.ccode_B == "002") & (sample.onset_A == 1)]
assert (ug["intervention"] > 0).any(), "USA→Guatemala intervention missing!"
print("✓ USA→Guatemala intervention present")

# USSR (364) in Afghanistan (700): gov-biased
af = sample[(sample.ccode_A == "700") & (sample.ccode_B == "364") & (sample.onset_A == 1)]
assert (af["intervention"] == 1).any(), "USSR→Afghanistan gov-biased intervention missing!"
print("✓ USSR→Afghanistan gov-biased intervention present")

# USA (002) in Afghanistan (700): opp-biased
af_us = sample[(sample.ccode_A == "700") & (sample.ccode_B == "002") & (sample.onset_A == 1)]
assert (af_us["intervention"] == 2).any(), "USA→Afghanistan opp-biased intervention missing!"
print("✓ USA→Afghanistan opp-biased intervention present")

# Non-onset rows must have NaN intervention
non_onset_ints = sample[sample["onset_A"] != 1]["intervention"].notna().sum()
assert non_onset_ints == 0, f"{non_onset_ints} non-onset rows have non-NaN intervention!"
print("✓ Non-onset rows all have NaN intervention")

# Intervention values are only 0, 1, 2 (or NaN)
valid_vals = {0.0, 1.0, 2.0, float('nan')}
all_vals   = set(sample["intervention"].dropna().unique())
assert all_vals <= {0, 1, 2}, f"Unexpected intervention values: {all_vals - {0,1,2}}"
print("✓ Intervention values are 0/1/2/NaN only")

print()
print("✓ Validation complete.")

DD_INT files produced: 25  (expected 25)



dd_int_1_1 — onset rows only:
  Unique onset country-years:           191
  Unique civil war hosts:               73
  Wars with any intervention:           44
  Wars with no intervention:            73
  Unique intervening states:            57
  Onset rows coded gov-biased (=1):     128
  Onset rows coded opp-biased (=2):     91
  Onset rows coded no-intervention (=0):25509

✓ USA→Guatemala intervention present
✓ USSR→Afghanistan gov-biased intervention present
✓ USA→Afghanistan opp-biased intervention present


✓ Non-onset rows all have NaN intervention
✓ Intervention values are 0/1/2/NaN only

✓ Validation complete.
